<h2>File paths and imports</h2>

This steps is important because it tells the program where it can access the files needed throughout the process.

You must specify 3 paths:

<li> <b>model_path</b>: body detection model
<li> <b>input_video_directory</b>: Folder containing the input videos
<li> <b>output_video_directory</b>: Folder where the treated videos will be redirected to

You can also mention videos that you don't want to be treated. To do so, simply indicate their name(s) without the extension in <b>ignore_S1</b> and <b>ignore_S2</b>.

In [1]:
import os
import sys
from ui_lib_bytetrack import *

# ==============================================================================
# CONFIGURATION: TUNE YOUR MODEL HERE
# ==============================================================================
# CHANGE THIS TO "v3_bytetrack" WHEN YOU RUN THIS NEW CODE
MODEL_VERSION = "v3_bytetrack" 

# Paths
model_path = "../Body_detection_model.pt"

input_video_directory = "../../../ChimpVideos/input"
output_video_directory = "../../../ChimpVideos/output"
temp_directory = f"{output_video_directory}/temp"
raw_text_output_directory = f"{temp_directory}/raw_output"
manual_annotations_directory = f"{input_video_directory}/manual_annotations"
treated_directory = f"{output_video_directory}/treated"
final_directory = f"{output_video_directory}/final"

# Videos to ignore per step
ignore_S1 = [
    # "12h41_short",
    "20241019 - 13h28",
    "20241019 - 14h29",
    "loma_mt",
]

ignore_S2 = [
    # "12h41_short",
    "20241019 - 13h28",
    "20241019 - 14h29",
    "loma_mt",
]

ignore_S3 = [
    # "12h41_short",
    "20241019 - 13h28",
    "20241019 - 14h29",
    "loma_mt",
]

# Helper: create required folders
for path in [
    input_video_directory,
    output_video_directory,
    temp_directory,
    raw_text_output_directory,
    manual_annotations_directory,
    treated_directory,
    final_directory,
]:
    os.makedirs(path, exist_ok=True)

# DeepSORT/YOLO setup
#nn_budget = None
#metric = nn_matching.NearestNeighborDistanceMetric("cosine", DEEPSORT_MAX_DIST, nn_budget)

#DeepSort = DeepSortTracker(
#    metric, 
#    max_iou_distance=0.7, 
#    max_age=DEEPSORT_MAX_AGE, 
#    n_init=DEEPSORT_N_INIT
#)

YOLOv8s = YOLO(model_path)
#Osnet = torchreid.models.build_model(name="osnet_x1_0", num_classes=751, pretrained=True)
#Osnet.eval()

def has_audio_stream(video_path: str) -> bool:
    """
    Optional: quick check to avoid mux when there is no audio.
    If your mux_audio already tolerates missing audio, you can skip this.
    """
    try:
        import subprocess, json
        probe_cmd = [
            "ffprobe",
            "-v", "error",
            "-select_streams", "a",
            "-show_entries", "stream=index",
            "-of", "json",
            video_path,
        ]
        out = subprocess.check_output(probe_cmd).decode("utf-8")
        data = json.loads(out)
        streams = data.get("streams", [])
        return len(streams) > 0
    except Exception:
        # Fallback: assume audio exists to keep behavior; mux_audio should handle errors gracefully
        return True

/home/ucl/ingi/trixen/ChimpRec/.venv/lib/python3.10/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


<h2>First step:</h2>

This step will process the input videos automatically. In other words, it will draw rectangles around each individuals and track them throughout the video.

In [2]:
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S1:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file_path = f"{manual_annotations_directory}/{video_name}.txt"
    try:
        with open(annotation_file_path, "x") as f:
            print(f"{video_name}.txt automatically created in {manual_annotations_directory}.")
    except FileExistsError:
        print(f"{video_name}.txt already present in {manual_annotations_directory}.")
    print()

    raw_txt_path = f"{raw_text_output_directory}/{video_name}.txt"

    # Tracking
    print(f"--- Processing {video_name} with {MODEL_VERSION} parameters ---")
    perform_tracking(
        input_video_path=full_video_path,
        output_text_file_path=raw_txt_path,
        detection_model=YOLOv8s,
        confidence_threshold=0.6 # ByteTrack handles low conf internally, so we can lower this back to 0.5
    )
    print(f"Annotations ready for video: {full_video_path}.\n")

    processed_with_audio = f"{temp_directory}/{video_name}-(temp)-audio.mp4"

    # Draw and mux (only one output, with audio when available)
    draw_bbox_from_file(
        file_path=raw_txt_path,
        input_video_path=full_video_path,
        output_video_path=processed_with_audio,
        annotation_type="bbox",
        draw_frame_count=True,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, processed_with_audio, processed_with_audio)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

20241019 - 13h28.mp4 ignored
12h41_short.txt already present in ../../../ChimpVideos/input/manual_annotations.

--- Processing 12h41_short with v3_bytetrack parameters ---


ByteTrack Processing:  96%|█████████▌| 1001/1043 [00:07<00:00, 133.52it/s]


Annotations ready for video: ../../../ChimpVideos/input/12h41_short.mp4.



Drawing annotations (12h41_short.mp4):  96%|█████████▌| 1001/1043 [00:14<00:00, 66.77it/s]
ffmpeg version 6.0 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 12.3.0 (GCC)
  configuration: --prefix=/opt/sw/arch/easybuild/2023a/software/FFmpeg/6.0-GCCcore-12.3.0 --enable-pic --enable-shared --enable-gpl --enable-version3 --enable-nonfree --cc=gcc --cxx=g++ --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libfreetype --enable-fontconfig --enable-libfribidi --enable-sdl2 --disable-htmlpages
  libavutil      58.  2.100 / 58.  2.100
  libavcodec     60.  3.100 / 60.  3.100
  libavformat    60.  3.100 / 60.  3.100
  libavdevice    60.  1.100 / 60.  1.100
  libavfilter     9.  3.100 /  9.  3.100
  libswscale      7.  1.100 /  7.  1.100
  libswresample   4. 10.100 /  4. 10.100
  libpostproc    57.  1.100 / 57.  1.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '../../../ChimpVideos/output/temp/12h41_short-(temp)-audio.mp4':
  Metadata:
    major_brand     : isom
    min

Adding audio...
Treatment done: ../../../ChimpVideos/input/12h41_short.mp4.

loma_mt.mp4 ignored
20241019 - 14h29.mp4 ignored


frame= 1001 fps=0.0 q=-1.0 Lsize=   99969kB time=00:00:20.01 bitrate=40925.7kbits/s speed=50.1x    
video:99479kB audio:472kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.018138%
[aac @ 0x47d2c0] Qavg: 674.301


<h2>Second step:</h2>

This final step will take into account your modifications to modify the output of the automated process.

<b>If you need to modify annotations previously created:</b> simply run this part of the code. In this case, there's no need to run the above cells.

In [3]:
# ---------- STEP 2: apply manual edits -> treated output (per-video folder) ----------
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S2:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file = f"{manual_annotations_directory}/{video_name}.txt"
    raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(
            f"Error: the manual annotation file related to the video <{full_video_path}> is not found. "
            f"It must be located at <{annotation_file}>."
        )
        continue

    # per-video output folder
    video_out_dir = os.path.join(treated_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)

    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-treated.txt")
    output_video_path = os.path.join(video_out_dir, f"{video_name}-treated.mp4")
    writer = data_writer(metadata_file_path)

    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path=metadata_file_path,
        input_video_path=full_video_path,
        output_video_path=output_video_path,
        annotation_type="bbox",
        draw_frame_count=True,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

20241019 - 13h28.mp4 ignored


Drawing annotations (12h41_short.mp4):  96%|█████████▌| 1001/1043 [00:11<00:00, 83.92it/s]
ffmpeg version 6.0 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 12.3.0 (GCC)
  configuration: --prefix=/opt/sw/arch/easybuild/2023a/software/FFmpeg/6.0-GCCcore-12.3.0 --enable-pic --enable-shared --enable-gpl --enable-version3 --enable-nonfree --cc=gcc --cxx=g++ --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libfreetype --enable-fontconfig --enable-libfribidi --enable-sdl2 --disable-htmlpages
  libavutil      58.  2.100 / 58.  2.100
  libavcodec     60.  3.100 / 60.  3.100
  libavformat    60.  3.100 / 60.  3.100
  libavdevice    60.  1.100 / 60.  1.100
  libavfilter     9.  3.100 /  9.  3.100
  libswscale      7.  1.100 /  7.  1.100
  libswresample   4. 10.100 /  4. 10.100
  libpostproc    57.  1.100 / 57.  1.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '../../../ChimpVideos/output/treated/12h41_short/12h41_short-treated.mp4':
  Metadata:
    major_brand     : is

Adding audio...
Treatment done: ../../../ChimpVideos/input/12h41_short.mp4.

loma_mt.mp4 ignored
20241019 - 14h29.mp4 ignored


frame= 1001 fps=0.0 q=-1.0 Lsize=   84522kB time=00:00:20.01 bitrate=34602.0kbits/s speed=50.4x    
video:84033kB audio:472kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.021453%
[aac @ 0x4a9c40] Qavg: 674.301


<h2> Third Step: </h2>

This final step will only be used to generate the arrows associated with the corresponding chimpanzees.<br>
<b>It's only to be performed when the manual annotations are 100% correct.</b>


In [ ]:
# ---------- STEP 3: final arrows/names -> final output (per-video folder) ----------
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S3:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file = f"{manual_annotations_directory}/{video_name}.txt"
    raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(
            f"Error: the manual annotation file related to the video <{full_video_path}> is not found. "
            f"It must be located at <{annotation_file}>."
        )
        continue

    # per-video output folder
    video_out_dir = os.path.join(final_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)

    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-final.txt")
    output_video_path = os.path.join(video_out_dir, f"{video_name}-final.mp4")
    writer = data_writer(metadata_file_path)

    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path=metadata_file_path,
        input_video_path=full_video_path,
        output_video_path=output_video_path,
        annotation_type="triangle",
        draw_frame_count=False,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")